# Indicadores y ajustes de retail: Preparación y controles de datos

Ejecuta la etapa anterior primero. Las cifras y conclusiones siguientes se calculan con los datos de esta ejecución; no edites manualmente sus valores. Consulta [metodología](../docs/metodologia.md) para interpretar resultados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("retail_kpi_analysis")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from retail_kpi_analysis.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from retail_kpi_analysis.analysis import run_stage
import pandas as pd
metrics, comment = run_stage(PROJECT_ROOT, "preparacion")
display(Markdown(comment))
scalars = {k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}
if scalars:
    display(pd.DataFrame(list(scalars.items()), columns=["Indicador", "Valor"]))
if "candidatos" in metrics:
    display(pd.DataFrame(metrics["candidatos"]))
if "baseline" in metrics and "modelo" in metrics:
    comparison = {key: {k: v for k, v in metrics[key].items() if not isinstance(v, (dict, list))} for key in ["baseline", "modelo"]}
    display(pd.DataFrame(comparison).T)
if "clasificacion" in metrics.get("modelo", {}):
    display(pd.DataFrame(metrics["modelo"]["clasificacion"]).T)
for image_name in ["eda.png", "evaluacion.png"]:
    image_path = PROJECT_ROOT / "reports/preparacion" / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))


# Indicadores y devoluciones de retail: preparacion

Se conservan **534,129 líneas válidas** después de retirar duplicados exactos, fechas/cantidades inválidas, precios no positivos y cantidades cero. Los ajustes negativos se mantienen; importes positivos y negativos se clasifican por signo. La moneda de la fuente es GBP. Los IDs de cliente faltantes permanecen en el análisis de facturación y se excluyen solo de recuentos de clientes.


,Indicador,Valor
0,filas_originales,541909
1,duplicados_retirados,5268
2,filas_invalidas_retiradas,2512
3,filas_preparadas,534129
4,sha256,f5385cbb54bbebf7196389109c6b0621faab0c304e3702...
5,fecha_min,2010-12-01 08:26:00
6,fecha_max,2011-12-09 12:50:00
7,cancelaciones_con_importe_positivo,0
8,moneda,GBP


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/01_preparacion_datos.json")
from retail_kpi_analysis.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))


[Abrir informe integral](../reports/informe_final/informe_final.md)